# Lab 1 — Frame, Split, Baseline
**SDA-AIE-111 · Day 1 · Hour 3 · 50 minutes · pairs**

**Objective.** Produce the framing canvas, a verified stratified split, and committed dummy + heuristic baselines for both Manafeth tasks (churn classification, basket-value regression).

**The rules this lab installs for the whole week:**
1. **Split first** — the test set is quarantined before any modelling decision.
2. **Baseline always** — every later model must beat the dummy *and* the domain heuristic.
3. **The timestamp test** — a feature is legal only if its value exists at prediction time.

> ⏱ Task timings are in each section header. If you fall behind, the checkpoint solution notebook lets you fast-forward.


In [1]:
# --- Course setup (identical in every lab) -----------------------------------
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

def find_repo_root() -> Path:
    """Locate the course repo root by looking for data/manafeth_customers.parquet."""
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "data" / "manafeth_customers.parquet").exists():
            return cand
    raise FileNotFoundError(
        "Course data not found — copy the Manafeth data package files into <repo>/data/")

ROOT = find_repo_root()
DATA = ROOT / "data"
sys.path.insert(0, str(ROOT / "src"))
print("repo root:", ROOT)

repo root: /home/claude/amlf/repo


## Task 1 — Data audit *(10 min)*

Load `manafeth_customers` and answer the three audit questions **in the markdown cell below the code**:
1. What is one row? (unit of analysis)
2. Is the target computable, and how balanced is it?
3. What can hurt us later? (missingness, weird distributions, suspicious columns)


In [2]:
customers = pd.read_parquet(DATA / "manafeth_customers.parquet")
print(customers.shape)
customers.head(3)

(48000, 19)


,customer_id,signup_date,snapshot_date,city,city_tier,device,payment_method,tenure_months,orders_per_month,avg_basket_sar,days_since_last_order,distinct_categories,promo_usage_rate,avg_rating,last_promo_used,churned_30d,refund_issued,support_ticket_after_snapshot,next_month_orders
0,C000001,2025-08-08,2025-11-01,Jeddah,2,Android,mada,2.8,2.51,84.79,12,7,0.322,4.15,RMDN25,0,0,0,2
1,C000002,2022-08-11,2025-11-01,Riyadh,1,Android,mada,38.7,1.58,148.06,4,6,0.114,5.00,NaN,0,0,0,2
2,C000003,2024-08-20,2025-11-01,Dammam,3,iOS,apple_pay,14.4,4.21,147.42,21,5,0.194,4.24,RMDN25,0,0,0,3


In [3]:
# Audit question 1: what is one row?
assert customers["customer_id"].is_unique, "unit-of-analysis violation: duplicate customers"
print("one row = one customer at the", customers["snapshot_date"].iloc[0].date(), "snapshot")

# Audit question 2: is the target computable and how balanced?
print(customers["churned_30d"].value_counts(normalize=True).round(3))

# Audit question 3: what can hurt us later?
print(customers.isna().mean().sort_values(ascending=False).head())

one row = one customer at the 2025-11-01 snapshot
churned_30d
0    0.86
1    0.14
Name: proportion, dtype: float64
avg_rating         0.313271
last_promo_used    0.220000
customer_id        0.000000
snapshot_date      0.000000
signup_date        0.000000
dtype: float64


**Your audit answers (fill in):**
- One row = …
- Churn rate = … → remember this number; it explains Module 4
- Missingness: `avg_rating` … (who is missing it, and why is that *informative*?)


## Task 2 — Framing canvas + leak hunt *(10 min)*

Complete `project/FRAMING.md` in pairs (template in the repo). Then use the **data dictionary** (`data/data_dictionary.md`) and the timestamp test — *would this value exist at prediction time?* — to flag the leaky columns. There are **three planted leaks**; find at least two.


In [4]:
# The three planted leaks — the timestamp test fails all of them:
#   refund_issued                  — refunds mostly happen AFTER the churn decision
#   support_ticket_after_snapshot  — the name confesses; recorded post-snapshot
#   next_month_orders              — "looks like a feature"; it IS the target in disguise
LEAKY = ["refund_issued", "support_ticket_after_snapshot", "next_month_orders"]

# Evidence for the sneakiest one: churners have next_month_orders == 0 BY CONSTRUCTION
print(customers.groupby("churned_30d")["next_month_orders"].mean().round(2))

churned_30d
0    2.97
1    0.00
Name: next_month_orders, dtype: float64


## Task 3 — The split *(10 min)*

Stratified, seeded, quarantined. The split happens **now**, in cell ~6 — not cell 30.


In [5]:
from sklearn.model_selection import train_test_split

TARGET = "churned_30d"
X = customers.drop(columns=[TARGET, "customer_id", *LEAKY])
y = customers[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,                 # preserve the 14% churn rate in both halves
    random_state=RANDOM_STATE,  # reproducibility is a course-wide rule
)
print(f"train {len(X_train):,} / test {len(X_test):,}")
print(f"churn: train {y_train.mean():.3f} | test {y_test.mean():.3f}  (stratification verified)")

# Quarantine: save the test partition; it is not touched again until Day 4 Hour 5.
(DATA / "splits").mkdir(exist_ok=True)
X_test.assign(**{TARGET: y_test}).join(customers[["customer_id"]]).to_parquet(
    DATA / "splits" / "test_quarantine.parquet")

train 38,400 / test 9,600
churn: train 0.140 | test 0.140  (stratification verified)


### 🔒 DO NOT OPEN
The file `data/splits/test_quarantine.parquet` is now **quarantined**. It is read exactly once, in Lab 7, for the final evaluation. Every decision between now and then uses the training data only. A test set consulted ten times is a validation set with a misleading name.


## Task 4 — Baselines *(10 min)*

Two baselines per task. The dummy defines the floor; the heuristic is what the business could deploy this afternoon for free.


In [6]:
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.metrics import average_precision_score, mean_absolute_error

# --- Churn: dummy floor
dummy = DummyClassifier(strategy="stratified", random_state=RANDOM_STATE).fit(X_train, y_train)
dummy_prauc = average_precision_score(y_train, dummy.predict_proba(X_train)[:, 1])
print(f"dummy PR-AUC: {dummy_prauc:.3f}   (= the churn rate; that is what 'no skill' looks like)")

# --- Churn: the CRM team's 21-day-silence heuristic, at the voucher budget
def recall_at_budget(y_true, scores, budget=0.20):
    """Recall among the top `budget` share ranked by score — the framing-canvas metric."""
    n = int(budget * len(y_true))
    top = np.argsort(-np.asarray(scores))[:n]
    return np.asarray(y_true)[top].sum() / np.asarray(y_true).sum()

heur_recall = recall_at_budget(y_train, X_train["days_since_last_order"].values)
print(f"heuristic (rank by silence) recall@20%: {heur_recall:.2f}")

dummy_recall = recall_at_budget(y_train, dummy.predict_proba(X_train)[:, 1])
print(f"dummy recall@20%: {dummy_recall:.2f}")

# --- Regression: dummy floor for basket value
y_train_basket = X_train["avg_basket_sar"]
dummy_reg = DummyRegressor(strategy="mean").fit(X_train, y_train_basket)
dummy_mae = mean_absolute_error(y_train_basket, dummy_reg.predict(X_train))
print(f"dummy MAE: {dummy_mae:.1f} SAR — the number to beat, in units the business understands")

dummy PR-AUC: 0.139   (= the churn rate; that is what 'no skill' looks like)
heuristic (rank by silence) recall@20%: 0.45
dummy recall@20%: 0.20
dummy MAE: 36.4 SAR — the number to beat, in units the business understands


## Task 5 — Scoreboard *(5 min)*

Post your numbers to the class scoreboard. **They must match your neighbours' to two decimals** — same seed, same data, same workflow. Divergent numbers always mean a workflow bug, and finding it publicly is the lesson.

| Model | PR-AUC | recall@20% | MAE (SAR) |
|---|---|---|---|
| dummy (stratified) | | | — |
| heuristic (21-day silence) | — | | — |
| dummy regressor (mean) | — | — | |

## Task 6 — Commit *(5 min)*

```
git add -A && git commit -m "feat(lab1): framing canvas, quarantined split, baselines"
```
